# Gun notlarini PDF olarak Drive'a yaz

Notlarin HTML kaynagi ve baski stili (`kbb/notlar/`) git deposunda durur.
Bu defter depoyu klonlar, her notu PDF'e cevirir ve `KBB_not_claude`
klasorune yazar. Uretilmis PDF'leri atlar, yani her calistirmada sadece
yeni notlar islenir.

Hucreleri sirayla calistir - duzenlemen gereken bir sey yok.


In [ ]:
# 1) Drive'i bagla
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) Araclari kur
!pip install -q weasyprint
!apt-get -qq install -y fonts-liberation > /dev/null
print("kurulum tamam")


In [ ]:
# 3) Depoyu cek ve notlari PDF'e cevir
#
# Notlarin HTML kaynagi ve baski stili git deposunda tutulur; bu hucre
# depoyu klonlar, kbb/notlar/ altindaki her HTML'i PDF'e cevirir ve
# KBB_not_claude klasorune yazar. Zaten var olan PDF'i atlar.

import glob, os, shutil, subprocess
from weasyprint import HTML

DEPO = "https://github.com/alpercil/Repository1"
YEREL = "/content/repo"

if os.path.exists(YEREL):
    shutil.rmtree(YEREL)
subprocess.run(["git", "clone", "--depth", "1", "-q", DEPO, YEREL], check=True)

# Hedef klasor: dogrulanmis yol, bulunamazsa ara
HEDEF = "/content/drive/MyDrive/PAÜ/KBB_not_claude"
if not os.path.isdir(HEDEF):
    adaylar = glob.glob("/content/drive/MyDrive/**/KBB_not_claude", recursive=True)
    if not adaylar:
        raise SystemExit("KBB_not_claude bulunamadi - HEDEF'i elle yaz")
    HEDEF = adaylar[0]
print("hedef klasor:", HEDEF)

kaynak = os.path.join(YEREL, "kbb", "notlar")
for html in sorted(glob.glob(os.path.join(kaynak, "*.html"))):
    ad = os.path.splitext(os.path.basename(html))[0]
    cikti = os.path.join(HEDEF, ad + ".pdf")
    if os.path.exists(cikti):
        print(f"  atlandi (zaten var): {ad}.pdf")
        continue
    HTML(html, base_url=kaynak).write_pdf(cikti)
    print(f"  uretildi: {ad}.pdf  {os.path.getsize(cikti)/1e6:.1f} MB")

print("\nBitti.")
